In [3]:
from aatm.data_models import MappedSourceConcept

path = 'vocabularies/SOURCE_TO_CONCEPT_MAP.csv'

mapped_source_concepts = MappedSourceConcept.from_csv(path)
len(mapped_source_concepts)

1463

In [4]:
from aatm.data_models import SourceConcept

path = 'vocabularies/SOURCE_TO_CONCEPT_MAP.csv'

mapped_source_concepts = SourceConcept.from_csv(path)
len(mapped_source_concepts)

1463

In [2]:
from aatm.data_models import SourceConcept, MappedSourceConcept


SourceConcept(
    source_code='tosse',
    source_concept_id='tosse',
    source_vocabulary_id='SNOMED',
    source_code_description='tosse',
    valid_start_date='2023-01-01',
    valid_end_date='2023-12-31',
    invalid_reason=None
)

SourceConcept(source_code='tosse', source_concept_id='tosse', source_vocabulary_id='SNOMED', source_code_description='tosse', valid_start_date='2023-01-01', valid_end_date='2023-12-31', invalid_reason=None)

In [7]:
MappedSourceConcept(
    source_code='tosse',
    source_concept_id=1234,
    source_vocabulary_id='SNOMED',
    source_code_description='tosse',
    target_concept_id='cough',
    target_vocabulary_id='SNOMED',
    domain_id='cough',
    valid_start_date='2023-01-01',
    valid_end_date='2023-12-31',
    invalid_reason=None
)

MappedSourceConcept(source_code='tosse', source_concept_id='1234', source_vocabulary_id='SNOMED', source_code_description='tosse', valid_start_date='2023-01-01', valid_end_date='2023-12-31', invalid_reason=None, target_concept_id='cough', target_vocabulary_id='SNOMED', domain_id='cough')

In [ ]:
%cd c:\Users\almei\Documents\GitHub\precision-data

from aatm.aio.translators import AsyncGeminiTranslator

translator = AsyncGeminiTranslator(model='gemini-2.5-flash')

await (['Selegiline 10 MG Oral Tablet by TEVA'] | translator)

c:\Users\almei\Documents\GitHub\precision-data


[Translation(text='cough')]

In [1]:
%cd c:\Users\almei\Documents\GitHub\precision-data
from aatm.translators import GeminiTranslator

translator = GeminiTranslator(model='gemini-2.5-flash')

['Selegiline 10 MG Oral Tablet by TEVA'] | translator

c:\Users\almei\Documents\GitHub\precision-data
Error while processing text 'Selegiline 10 MG Oral Tablet by TEVA' (type: <class 'str'>): the JSON object must be str, bytes or bytearray, not NoneType. Retrying... (1/3)
Malformed response: "sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=None create_time=None model_version='gemini-2.5-flash' prompt_feedback=GenerateContentResponsePromptFeedback(
  block_reason=<BlockedReason.PROHIBITED_CONTENT: 'PROHIBITED_CONTENT'>
) response_id='P-N-aYOxEeC-qtsPoKTyyAc' usage_metadata=GenerateContentResponseUsageMetadata(
  prompt_token_count=22,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=22
    ),
  ],
  total_token_count=22
) automatic_function_calling_history=[] parsed=None"
Error while processing text 'Selegiline 10 MG Oral Tablet by TEVA' (type: <class 'str'>): the JSON object must be str, bytes or bytearray, not NoneType. Retrying... (2/3)
Malformed respons

[Translation(text='Selegiline 10 MG Oral Tablet by TEVA')]

In [9]:
from pydantic import BaseModel
import google.genai as genai
from google.genai import types
import os

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

class Translation(BaseModel):
    text: str

response = client.models.generate_content(
    model='gemini-3-flash-preview',
    contents='Translate the following text into English: "MFOLFOX"',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=Translation,
    ),
)
print(response.text)

{"text":"mFOLFOX"}


In [17]:
import json
import asyncio
from typing import List

class PipelineBaseClass:
    def __or__(self, other):
        return other(self)

    def __ror__(self, other):
        return self(other)

class GeminiTranslator(PipelineBaseClass):
    def __init__(self, model: str, *args, **kwargs):
        self.client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
        self.model = model

    async def __call__(self, text: str|List[str]) -> List[Translation]:
        if isinstance(text, str):
            text = [text]

        async with asyncio.TaskGroup() as tg:
            tasks = [
                tg.create_task(
                    client.aio.models.generate_content(
                        model=self.model,
                        contents=f'Translate the following text into English: "{t}"',
                        config=types.GenerateContentConfig(
                            response_mime_type='application/json',
                            response_schema=Translation,
                        ),
                    )
                )
                for t in text
            ]

        results = [t.result() for t in tasks]
        results = [json.loads(r.text) for r in results]
        return [Translation(**r) for r in results]

        # response = await client.aio.models.generate_content(
        #     model=self.model,
        #     contents=f'Translate the following text into English: "{text}"',
        #     config=types.GenerateContentConfig(
        #         response_mime_type='application/json',
        #         response_schema=Translation,
        #     ),
        # )
        # return Translation(**json.loads(response.text))

translator = GeminiTranslator(model='gemini-2.5-flash')

await ('dor de barriga' | translator)

[Translation(text='stomach ache')]

In [4]:
from aatm.aio.translators import AsyncGeminiTranslator

translator = AsyncGeminiTranslator(model='gemini-2.5-flash')

await (['tosse', 'febre', 'dor abdominal', 'diarreia'] | translator)

[Translation(text='cough'),
 Translation(text='fever'),
 Translation(text='abdominal pain'),
 Translation(text='diarrhea')]

In [22]:
'dor de barriga' | translator

Translation(text='stomach ache')

In [1]:
from aatm.translators import GeminiTranslator

translator = GeminiTranslator(model='gemini-2.5-flash')

In [5]:
%%timeit
'dor de barriga' | translator

979 ms ± 367 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit
translator('dor de barriga')

901 ms ± 105 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
